In [ ]:
import urllib.request

url = 'https://box.fu-berlin.de/s/P4LZ6JYBnbqdewf/download/age_pt.zip'
urllib.request.urlretrieve(url, "data.zip")

In [ ]:
!mkdir data

In [ ]:
from zipfile import ZipFile
ZipFile("data.zip").extractall("./data")

# Preparing data folders for Yolo

In [ ]:
import os
import shutil
import pandas as pd
from sklearn.model_selection import train_test_split

# ---------------- CONFIG ----------------
IMAGES_SRC = "data/age_pt/train/2"
OUTPUT_ROOT = "yolo"

TRAIN_RATIO = 0.7
RANDOM_STATE = 42

# ----------------------------------------

def ensure_dirs():
    for split in ["train", "val"]:
        os.makedirs(os.path.join(OUTPUT_ROOT, "images", split), exist_ok=True)
        os.makedirs(os.path.join(OUTPUT_ROOT, "labels", split), exist_ok=True)

def write_yolo_label(df, label_path):
    """
    Write a YOLO .txt file from rows belonging to one image
    """
    lines = []
    for _, r in df.iterrows():
        line = f"{str(r.l)} {r.x_center:.6f} {r.y_center:.6f} {r.w_norm:.6f} {r.h_norm:.6f}"
        lines.append(line)

    with open(label_path, "w") as f:
        f.write("\n".join(lines))

def populate_yolo_dirs(df):
    ensure_dirs()

    # unique images
    images = df["file_name"].unique()
    print(images)

    train_imgs, val_imgs = train_test_split(
        images,
        train_size = TRAIN_RATIO,
        random_state = RANDOM_STATE,
        shuffle = True
    )

    splits = {
        "train": train_imgs,
        "val": val_imgs
    }

    for split, image_list in splits.items():
        for img_name in image_list:
            src_img = os.path.join(IMAGES_SRC, img_name)
            dst_img = os.path.join(OUTPUT_ROOT, "images", split, img_name)

            # copy image
            shutil.copy2(src_img, dst_img)

            # labels
            label_name = os.path.splitext(img_name)[0] + ".txt"
            label_path = os.path.join(OUTPUT_ROOT, "labels", split, label_name)

            img_rows = df[df["file_name"] == img_name]
            write_yolo_label(img_rows, label_path)

    print("YOLO dataset populated successfully.")

In [ ]:
def convert_to_yolo(df, img_w, img_h):
    df = df.copy()

    df["x_center"] = (df["x"] + df["w"] / 2) / img_w
    df["y_center"] = (df["y"] + df["h"] / 2) / img_h
    df["w_norm"] = df["w"] / img_w
    df["h_norm"] = df["h"] / img_h

    return df


## Loading annotations (coordinates of bounding boxes)

In [ ]:
d = pd.read_csv("bboxes.csv")

# replacing Kücken and Eltern

d1 = d.replace({"l": r"Kuecken"}, {"l": 0}, regex=True)
d1 = d1.replace({"l": r"Eltern"}, {"l": 1}, regex=True)

d_yolo = convert_to_yolo(d, 948, 656)
populate_yolo_dirs(d_yolo)

In [ ]:
!pip install ultralytics

In [ ]:
from ultralytics import YOLO

def train():
    model = YOLO("yolov8n.pt")  # n, s, m, l, x

    model.train(
        data="dataset/data.yaml",
        epochs=100,
        imgsz=640,
        batch=16,
        device=0,          # 0 for GPU, "cpu" for CPU
        workers=8,
        project="runs",
        name="yolov8_custom"
    )

In [ ]:
train()

In [ ]:
# Load trained model
model = YOLO("runs/detect/runs/yolov8_custom4/weights/best.pt")

# Run inference
results = model('data/age_pt/train/2/' + 
                flist[10], conf=0.25)

# Show prediction
results[0].show()